[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Boyu-Zhang-UOI/minimind-colab/blob/main/notebooks/03_Pretraining.ipynb)

# Module 3: Pretraining — From Raw Text to Language Model

In [ ]:
# @title 🔧 Environment Setup (Run this first!)
import os

# Check GPU
gpu_info = !nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null || echo "No GPU"
print(f"GPU: {gpu_info[0]}")

if not os.path.exists('minimind-colab'):
    !git clone https://github.com/Boyu-Zhang-UOI/minimind-colab.git
os.chdir('minimind-colab')

!pip install -q transformers==4.46.3 datasets==3.6.0 jinja2==3.1.2 tokenizers matplotlib

import sys
sys.path.insert(0, '.')
print("✅ Setup complete!")

## 📚 Overview

In this module we complete the MiniMind model architecture and run our first **pretraining** loop. Pretraining teaches the model to predict the next token given all previous tokens — the fundamental capability behind every LLM.

**What we will cover:**
1. SiLU-gated Feed-Forward Network (FFN)
2. Mixture of Experts (MoE)
3. Full model assembly (`MiniMindBlock` → `MiniMindModel` → `MiniMindForCausalLM`)
4. The pretraining data pipeline (`PretrainDataset`)
5. Training utilities (learning rate schedule, checkpointing)
6. The pretraining loop (mixed precision, gradient accumulation)
7. Testing generation from our trained model

**Key source files:** `model/model_minimind.py` (lines 134–279), `dataset/lm_dataset.py`, `trainer/trainer_utils.py`, `trainer/train_pretrain.py`

## 1. SiLU-Gated Feed-Forward Network

The FFN in MiniMind uses a **SiLU-gated** architecture (SwiGLU variant), standard in modern LLMs like LLaMA:

$$\text{FFN}(x) = W_{\text{down}} \cdot \left(\text{SiLU}(W_{\text{gate}} \cdot x) \odot W_{\text{up}} \cdot x\right)$$

Three separate linear projections:
- `gate_proj`: Controls which features to activate (via SiLU)
- `up_proj`: Projects input to intermediate space
- `down_proj`: Projects back to hidden size

The intermediate size is `ceil(hidden_size × π / 64) × 64 = 2304` for hidden_size=768.

In [ ]:
import torch
import sys
sys.path.insert(0, '.')
from model.model_minimind import MiniMindConfig, MiniMindForCausalLM, FeedForward

config = MiniMindConfig(hidden_size=768, num_hidden_layers=8)
ffn = FeedForward(config)

# Show structure
print("=== SiLU-Gated FFN Structure ===")
print(f"Gate proj: {ffn.gate_proj.weight.shape}  (hidden → intermediate)")
print(f"Up proj:   {ffn.up_proj.weight.shape}  (hidden → intermediate)")
print(f"Down proj: {ffn.down_proj.weight.shape}  (intermediate → hidden)")
print(f"\nIntermediate size: {config.intermediate_size}")
print(f"FFN params: {sum(p.numel() for p in ffn.parameters()):,}")

# Forward pass
x = torch.randn(1, 10, 768)
out = ffn(x)
print(f"\nInput shape:  {x.shape}")
print(f"Output shape: {out.shape}")

## 2. Mixture of Experts (MoE)

MoE replaces the single FFN with **multiple expert** FFNs and a **router** that selects which expert(s) to use per token:

```
Input → Router → [Expert 0, Expert 1, Expert 2, Expert 3] → Weighted Sum → Output
                     ↑ selected (top-k=1)
```

Key concepts:
- **Router**: Linear layer + softmax → probability over experts
- **Top-k selection**: Only `k=1` expert is active per token (sparse)
- **Auxiliary loss**: Encourages balanced expert utilization
- **Parameter count**: 4× FFN params total, but only 1× active per token

In [ ]:
from model.model_minimind import MOEFeedForward

moe_config = MiniMindConfig(
    hidden_size=768,
    num_hidden_layers=8,
    use_moe=True,
    num_experts=4,
    num_experts_per_tok=1
)
moe_ffn = MOEFeedForward(moe_config)

print("=== MoE FFN Structure ===")
print(f"Router: {moe_ffn.gate.weight.shape}  (hidden → num_experts)")
print(f"Number of experts: {len(moe_ffn.experts)}")
print(f"Expert FFN params: {sum(p.numel() for p in moe_ffn.experts[0].parameters()):,}")
print(f"Total MoE params: {sum(p.numel() for p in moe_ffn.parameters()):,}")
print(f"Active params/token: {sum(p.numel() for p in moe_ffn.experts[0].parameters()) + moe_ffn.gate.weight.numel():,}")

# Forward pass
x = torch.randn(2, 10, 768)
moe_ffn.train()
out = moe_ffn(x)
print(f"\nInput shape:  {x.shape}")
print(f"Output shape: {out.shape}")
print(f"Auxiliary loss: {moe_ffn.aux_loss.item():.6f}")

## 3. Full Model Assembly

The complete MiniMind model stacks everything together:

```
Input IDs → Embedding → [MiniMindBlock × 8] → RMSNorm → LM Head → Logits
                              ↓
                    Attention + FFN (or MoE)
                    with pre-norm residuals
```

**Weight tying**: The embedding matrix and LM head share the same weights, saving ~5M parameters.

**Loss**: Standard cross-entropy between predicted logits and shifted labels (next-token prediction).

In [ ]:
from transformers import AutoTokenizer

model = MiniMindForCausalLM(config)
tokenizer = AutoTokenizer.from_pretrained('./model')

# Parameter count
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,} ({total_params/1e6:.1f}M)")

# Verify weight tying
print(f"\nWeight tying: {model.model.embed_tokens.weight is model.lm_head.weight}")

# Forward pass with loss
input_ids = torch.randint(0, config.vocab_size, (2, 32))
labels = input_ids.clone()
output = model(input_ids, labels=labels)
print(f"\nInput shape:  {input_ids.shape}")
print(f"Logits shape: {output.logits.shape}")
print(f"Loss: {output.loss.item():.4f}")
print(f"Aux loss: {output.aux_loss.item():.4f}")

## 4. Pretraining Data Pipeline

The `PretrainDataset` class loads raw text and prepares it for causal language modeling:

| Step | Operation | Example |
|------|-----------|--------|
| 1 | Load JSONL | `{"text": "Machine learning is..."}` |
| 2 | Tokenize | `[5123, 234, 1567, ...]` |
| 3 | Add special tokens | `[BOS] + tokens + [EOS]` |
| 4 | Pad to max_length | `tokens + [PAD] × N` |
| 5 | Create labels | Copy of input_ids; PAD → -100 |

The label `-100` tells PyTorch's cross-entropy to **ignore** padding tokens during loss computation.

In [ ]:
from dataset.lm_dataset import PretrainDataset

# Create a tiny sample dataset
import json, tempfile, os
sample_data = [
    {"text": "Machine learning is a subset of artificial intelligence that enables systems to learn from data."},
    {"text": "Natural language processing helps computers understand human language."},
    {"text": "Deep learning uses neural networks with many layers to learn complex patterns."},
    {"text": "Transformers revolutionized NLP with the self-attention mechanism."},
    {"text": "Large language models can generate coherent text by predicting the next token."},
]

data_path = '/tmp/sample_pretrain.jsonl'
with open(data_path, 'w') as f:
    for item in sample_data:
        f.write(json.dumps(item) + '\n')

dataset = PretrainDataset(data_path, tokenizer, max_length=64)
print(f"Dataset size: {len(dataset)}")

# Inspect a sample
input_ids, labels = dataset[0]
print(f"\nSample 0:")
print(f"  Input IDs shape: {input_ids.shape}")
print(f"  Labels shape:    {labels.shape}")
print(f"  BOS token (ID {tokenizer.bos_token_id}): position 0")
print(f"  PAD positions: {(input_ids == tokenizer.pad_token_id).sum().item()} tokens")
print(f"  Masked labels (-100): {(labels == -100).sum().item()} positions")
print(f"\n  First 20 tokens: {input_ids[:20].tolist()}")
print(f"  First 20 labels: {labels[:20].tolist()}")
print(f"\n  Decoded: {tokenizer.decode(input_ids[input_ids != tokenizer.pad_token_id])}")

## 5. Training Utilities

Before the training loop, we need three key utilities from `trainer/trainer_utils.py`:

### Cosine Learning Rate Schedule
```python
def get_lr(current_step, total_steps, lr):
    return lr * (0.1 + 0.45 * (1 + cos(π * current_step / total_steps)))
```
Starts at ~55% of max LR, rises to max, then decays to ~10%.

### Model Initialization
`init_model()` loads config, tokenizer, creates the model, and optionally loads a pretrained checkpoint.

### Checkpointing
`lm_checkpoint()` saves model weights, optimizer state, epoch, and step for training resumption.

In [ ]:
import matplotlib.pyplot as plt
import math

def get_lr(current_step, total_steps, lr):
    return lr * (0.1 + 0.45 * (1 + math.cos(math.pi * current_step / total_steps)))

# Visualize the learning rate schedule
total_steps = 1000
lr_base = 5e-4
steps = range(total_steps)
lrs = [get_lr(s, total_steps, lr_base) for s in steps]

plt.figure(figsize=(10, 4))
plt.plot(steps, lrs, 'b-', linewidth=2)
plt.xlabel('Training Step')
plt.ylabel('Learning Rate')
plt.title('Cosine Annealing Learning Rate Schedule')
plt.grid(True, alpha=0.3)
plt.axhline(y=lr_base, color='r', linestyle='--', alpha=0.5, label=f'Base LR = {lr_base}')
plt.legend()
plt.tight_layout()
plt.show()
print(f"Initial LR: {lrs[0]:.6f}")
print(f"Peak LR: {max(lrs):.6f}")
print(f"Final LR: {lrs[-1]:.6f}")

## 6. The Pretraining Loop

The core training loop in `trainer/train_pretrain.py` follows this pattern:

```
for each batch:
    1. Compute learning rate (cosine schedule)
    2. Forward pass with mixed precision (bfloat16)
    3. Compute loss = CE_loss + aux_loss (MoE)
    4. Scale loss for gradient accumulation
    5. Backward pass
    6. Every N steps: clip gradients → optimizer step
    7. Log metrics and save checkpoints
```

**Key hyperparameters for pretraining:**
| Parameter | Value | Purpose |
|-----------|-------|--------|
| `learning_rate` | 5e-4 | Base learning rate |
| `batch_size` | 32 | Samples per batch |
| `max_seq_len` | 340 | Maximum sequence length |
| `accumulation_steps` | 8 | Gradient accumulation |
| `grad_clip` | 1.0 | Max gradient norm |
| `dtype` | bfloat16 | Mixed precision |

In [ ]:
import time
from torch import optim
from contextlib import nullcontext

# Create model and move to device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = MiniMindForCausalLM(config).to(device)
print(f"Training on: {device}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

# Create dataset and dataloader
dataset = PretrainDataset(data_path, tokenizer, max_length=64)
from torch.utils.data import DataLoader
loader = DataLoader(dataset, batch_size=2, shuffle=True)

# Optimizer and mixed precision
optimizer = optim.AdamW(model.parameters(), lr=5e-4, weight_decay=0.01)
dtype = torch.bfloat16 if device == 'cuda' and torch.cuda.is_bf16_supported() else torch.float32
autocast_ctx = torch.cuda.amp.autocast(dtype=dtype) if device == 'cuda' else nullcontext()
scaler = torch.cuda.amp.GradScaler(enabled=(dtype == torch.float16))

# Training loop
model.train()
num_epochs = 3
accumulation_steps = 1
total_steps = num_epochs * len(loader)
losses = []

print(f"\n{'='*60}")
print(f"Starting pretraining: {num_epochs} epochs, {len(loader)} steps/epoch")
print(f"{'='*60}\n")

for epoch in range(num_epochs):
    epoch_loss = 0
    for step, (input_ids, labels) in enumerate(loader, 1):
        input_ids = input_ids.to(device)
        labels = labels.to(device)

        # Cosine LR schedule
        global_step = epoch * len(loader) + step
        lr = get_lr(global_step, total_steps, 5e-4)
        for pg in optimizer.param_groups:
            pg['lr'] = lr

        # Forward with mixed precision
        with autocast_ctx:
            res = model(input_ids, labels=labels)
            loss = res.loss + res.aux_loss
            loss = loss / accumulation_steps

        # Backward
        scaler.scale(loss).backward()

        if step % accumulation_steps == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        current_loss = loss.item() * accumulation_steps
        epoch_loss += current_loss
        losses.append(current_loss)

    avg_loss = epoch_loss / len(loader)
    print(f"Epoch {epoch+1}/{num_epochs} | Avg Loss: {avg_loss:.4f} | LR: {lr:.6f}")

print(f"\n✅ Pretraining complete! Final loss: {losses[-1]:.4f}")

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(losses, 'b-', alpha=0.7, linewidth=1)
plt.xlabel('Training Step')
plt.ylabel('Loss')
plt.title('Pretraining Loss Curve')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Testing Generation

Even with minimal training, we can test the model's `generate()` method. The generation loop:

1. **Temperature scaling**: `logits / T` — higher T = more random
2. **Top-k filtering**: Keep only top 50 tokens by probability
3. **Top-p (nucleus) sampling**: Keep tokens until cumulative prob ≥ 0.95
4. **Repetition penalty**: Penalize already-generated tokens
5. **KV-Cache**: Reuse computed K, V from previous tokens

With so little training data, output will be mostly random — but it demonstrates the full pipeline works!

In [ ]:
model.eval()

prompts = ["Machine learning", "The transformer", "Language models"]
for prompt in prompts:
    inputs = tokenizer(tokenizer.bos_token + prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        output_ids = model.generate(
            inputs.input_ids,
            attention_mask=inputs.attention_mask,
            max_new_tokens=30,
            temperature=0.85,
            top_p=0.95,
            do_sample=True
        )
    generated = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    print(f"Prompt: '{prompt}'")
    print(f"Output: '{generated}'\n")

## ✏️ Exercises

1. **Compare Dense vs MoE**: Create a MoE model with `use_moe=True` and compare its parameter count to the dense model. How many parameters are "active" per token?

2. **Vary sequence length**: Try `max_length=128` and `max_length=340` in the PretrainDataset. How does this affect training speed and memory usage?

3. **Gradient accumulation**: Set `accumulation_steps=4` and reduce `batch_size` accordingly. Verify the effective batch size stays the same.

4. **Learning rate**: Try `learning_rate=1e-3` (too high) and `learning_rate=1e-5` (too low). Observe how the loss curve changes.

## 📝 Summary

In this module we covered:

- **SiLU-Gated FFN** — The `gate_proj` / `up_proj` / `down_proj` pattern used in modern LLMs
- **Mixture of Experts** — Sparse routing to multiple expert FFNs with auxiliary loss
- **Full model assembly** — Embedding → Transformer blocks → LM Head with weight tying
- **PretrainDataset** — JSONL text → tokenized sequences with BOS/EOS/PAD and label masking
- **Training utilities** — Cosine LR schedule, checkpointing, mixed precision
- **Pretraining loop** — Forward → loss → backward → clip → step with gradient accumulation
- **Generation testing** — Autoregressive sampling with temperature, top-k, and top-p

🔜 **Next module:** Supervised Fine-Tuning (SFT) and LoRA — teaching the model to follow instructions!